# Tree-based models comparison

The goal of this notebook is to display three tree-based algorithms on a single 1 dimension regression problem :
1. Regression Tree
2. Random Forest
3. Gradient Boosting

To make it interactive, we will use Jupyter Notebook widgets. We first need to build the regression problem :

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

N_POINTS = 500
RANDOM_SEED = 42


def true_function(x):
    return np.cos(x) - np.tanh(x)


def generate_base_data(n_points=N_POINTS, x_min=0, x_max=8, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    x = np.linspace(x_min, x_max, n_points)
    raw_noise = rng.normal(size=n_points)
    return x, raw_noise


X, RAW_NOISE = generate_base_data()
TRAIN_IDX, TEST_IDX = train_test_split(
    np.arange(N_POINTS), test_size=0.3, random_state=RANDOM_SEED
)


def make_noisy_target(noise_std):
    return true_function(X) + noise_std * RAW_NOISE

We have define three functions :
1. `true_function` : the function we will approximate, our target
2. `generate_base_data` : generate the feature $x$
3. `generate_noisy_target` : generate the target $y$ and we make it noisy to visualize how each methods handles noisy data

Let's move to the algorithms next.

In [ ]:
def get_model_configs(max_depth, n_estimators, learning_rate):
    configurations = dict()
    configurations["Decision Tree"] = (
        DecisionTreeRegressor, 
        {"max_depth": max_depth, "random_state": RANDOM_SEED},
    )

    configurations["Random Forest"] = (
        RandomForestRegressor,
        {"max_depth": max_depth, "n_estimators": n_estimators, "random_state": RANDOM_SEED},
    )

    configurations["Boosting"] = (
        HistGradientBoostingRegressor,
        {"max_depth": max_depth, "max_iter": n_estimators, "learning_rate": learning_rate, "random_state": RANDOM_SEED}
    )
    return configurations

def fit_and_predict(model_cls, params, x_train, y_train, x_full):
    model = model_cls(**params).fit(pd.DataFrame({"x": x_train}), y_train)
    return model.predict(pd.DataFrame({"x": x_full}))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

We built again three functions : 
1. `get_model_configs` : each models having its own configuation, we build each of them separatly
2. `fit_and_predict` : as we have three models, we fit and make prediction through this function
3. `rmse` : compute the RMSE between two vectors

Having the data and the models, we have the plotting still.

In [ ]:
def plot_model_panel(ax, name, y_noisy, y_true, y_pred):
    ax.plot(X, y_true, color="blue", linewidth=2, label="True function")
    ax.scatter(X[TRAIN_IDX], y_noisy[TRAIN_IDX], color="gray", alpha=0.2, s=15, label="Train")
    ax.scatter(X[TEST_IDX], y_noisy[TEST_IDX], color="orange", alpha=0.2, s=15, label="Test")
    ax.plot(X, y_pred, color="red", linewidth=2, label="Prediction")

    train_rmse = rmse(y_noisy[TRAIN_IDX], y_pred[TRAIN_IDX])
    test_rmse = rmse(y_noisy[TEST_IDX], y_pred[TEST_IDX])
    ax.set_title(f"{name}\nTrain RMSE: {train_rmse:.3f} | Test RMSE: {test_rmse:.3f}")
    ax.legend(loc="upper right", fontsize=8)

def update_plot(noise_std, max_depth, n_estimators, learning_rate):
    y_noisy = make_noisy_target(noise_std)
    y_true = true_function(X)
    x_train, y_train = X[TRAIN_IDX], y_noisy[TRAIN_IDX]

    configs = get_model_configs(max_depth, n_estimators, learning_rate)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, (name, (model_cls, params)) in zip(axes, configs.items()):
        y_pred = fit_and_predict(model_cls, params, x_train, y_train, X)
        plot_model_panel(ax, name, y_noisy, y_true, y_pred)

    plt.tight_layout()
    plt.show()

And now we can interact !

In [10]:
interact(
    update_plot,
    noise_std=widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, description="Noise std"),
    max_depth=widgets.IntSlider(value=4, min=1, max=10, step=1, description="Max depth"),
    n_estimators=widgets.IntSlider(value=10, min=1, max=50, step=1, description="N estimators"),
    learning_rate=widgets.FloatSlider(value=0.3, min=0.01, max=1.0, step=0.01, description="Learning rate"),
)

interactive(children=(FloatSlider(value=0.2, description='Noise std', max=1.0, step=0.05), IntSlider(value=4, …

<function __main__.update_plot(noise_std, max_depth, n_estimators, learning_rate)>